# Phase 4: Causal Patching — Activation Patching

**Experimental protocol:**
1. Run **counterfactual prompt** → save hidden state at Layer 10 (last token)
2. Run **clean prompt** normally → save the baseline answer
3. Run **clean prompt again**, but REPLACE Layer 10's hidden state (last token of prefill) with the CF state harvested in step 1
4. Measure whether the answer changed

If the answer changes in ≥50% of cases, Layer 10 is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
TARGET_LAYER = 10

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
def extract_answer_tag(text):
    """Pull out just the <answer>...</answer> part."""
    import re
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if m:
        return m.group(1).strip().lower()
    # Fallback: last non-empty line
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    return lines[-1].lower() if lines else ""

print(f"Running causal patching (Entity-Level) on {len(pairs)} pairs at Layer {TARGET_LAYER}...")

# We will do DENOISING: Patching CLEAN state into CORRUPTED prompt to restore correctness.
n_changed = 0       # Answer changed from wrong to correct (successful denoising)
n_already_correct = 0 
n_total = 0
examples_log = []

for i, (clean, cf) in enumerate(tqdm(pairs, desc="Causal Patching")):
    # 1. Get the text up to the end of Hop 1
    # We parse the generated_cot to extract Hop 1 text exactly as the model generated it.
    import re
    def get_hop1(cot):
        m = re.search(r'(<hop1>.*?</hop1>)', cot, re.DOTALL)
        return m.group(1) if m else ""
    
    clean_h1_text = get_hop1(clean["generated_cot"])
    cf_h1_text = get_hop1(cf["generated_cot"])
    
    if not clean_h1_text or not cf_h1_text:
        continue
        
    clean_full_prompt = clean["prompt"] + clean_h1_text
    cf_full_prompt = cf["prompt"] + cf_h1_text
    
    # 2. HARVEST the CLEAN state at the end of Hop 1
    cf_state = {}
    def harvest_hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        # Grab the very last token of the prefill (which is </hop1>)
        cf_state['vec'] = hidden[0, -1, :].clone().detach()
        return output
    
    clean_inputs = tokenizer(clean_full_prompt, return_tensors="pt").to(model.device)
    harvest_handle = model.model.layers[TARGET_LAYER].register_forward_hook(harvest_hook)
    with torch.no_grad():
        model(**clean_inputs)
    harvest_handle.remove()
    clean_vec = cf_state['vec']
    
    # 3. Get BASELINE answer for CF (Should be WRONG, as it's a hallucination)
    cf_inputs = tokenizer(cf_full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        baseline_out = model.generate(**cf_inputs, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id)
    baseline_text = tokenizer.decode(baseline_out[0][cf_inputs.input_ids.shape[1]:], skip_special_tokens=True)
    baseline_ans = extract_answer_tag(baseline_text)
    
    expected_clean_ans = clean.get("predicted_answer", clean.get("answer", "")).lower()
    
    # 4. INJECT the CLEAN state into the CF generation
    injected = [False]
    def inject_hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        rest = output[1:] if isinstance(output, tuple) else None
        
        # We only inject ONCE during the prefill of the CF prompt
        if not injected[0] and hidden.shape[1] > 1:
            hidden = hidden.clone()
            hidden[0, -1, :] = clean_vec  # Inject clean vector at </hop1>
            injected[0] = True
            
        if rest is not None:
            return (hidden,) + rest
        return hidden
    
    inject_handle = model.model.layers[TARGET_LAYER].register_forward_hook(inject_hook)
    with torch.no_grad():
        corrupted_out = model.generate(**cf_inputs, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id)
    inject_handle.remove()
    corrupted_text = tokenizer.decode(corrupted_out[0][cf_inputs.input_ids.shape[1]:], skip_special_tokens=True)
    corrupted_ans = extract_answer_tag(corrupted_text)
    
    # Check if the injection RESTORED the correct answer!
    # A successful denoising changes the CF's wrong answer into the Clean's correct answer.
    restored = False
    if expected_clean_ans in corrupted_ans and expected_clean_ans not in baseline_ans:
        restored = True
        n_changed += 1
        
    n_total += 1
    
    examples_log.append({
        "clean_q": clean["question"],
        "cf_q": cf["question"],
        "baseline_ans": baseline_ans,
        "corrupted_ans": corrupted_ans,
        "restored": restored
    })
    
    if i < 3:
        print(f"\n--- Example {i+1} ---")
        print(f"CF Q:          {cf['question']}")
        print(f"Clean Ans:     {expected_clean_ans}")
        print(f"CF Baseline:   {baseline_ans}")
        print(f"CF Patched:    {corrupted_ans}")
        print(f"Restored?      {'✅ YES' if restored else '❌ NO'}")

rate = n_changed / n_total * 100 if n_total > 0 else 0
print(f"\n{'='*40}")
print(f"=== CAUSAL PATCHING (DENOISING) RESULTS ===")
print(f"{'='*40}")
print(f"Total pairs tested:       {n_total}")
print(f"Answers restored:         {n_changed}")
print(f"Denoising Success Rate:   {rate:.1f}%")
print(f"")
print(f"Interpretation:")
if rate >= 50:
    print(f"  ✅ STRONG CAUSAL EFFECT: Layer {TARGET_LAYER} state is sufficient to")
    print(f"     restore reasoning. Clean state fixes {rate:.0f}% of hallucinations.")
elif rate >= 20:
    print(f"  ⚠️  MODERATE CAUSAL EFFECT: Layer {TARGET_LAYER} has partial influence.")
else:
    print(f"  ❌ WEAK CAUSAL EFFECT: Single-layer denoising is insufficient.")
